# 🔁 Desafio 2: Ingestao Perene de Dados do YouTube
## Do cientista que coleta ao engenheiro que **governa** o dado

**Disciplina:** Linguagens de Programacao para Engenharia de Dados
**Instituicao:** UNIFOR · **Professor:** Cassio Pinheiro

---

No desafio anterior voce **coletou** transcricoes colando IDs de video na mao.
Isso e o trabalho do cientista de dados explorando uma fonte. Funciona uma vez.

O engenheiro de dados pensa diferente. A pergunta nao e *"como pego este video?"*
e sim *"como garanto que TODO video novo deste canal entre no pipeline,
sozinho, para sempre, sem eu colar nada?"*.

Este notebook ensina os **quatro conceitos** que transformam coleta manual em
**ingestao perene**, e depois aponta para o projeto-script de producao
(`youtube_ingestor/`) que implementa tudo pronto para rodar numa VPS.


## Os 4 pilares da ingestao perene

| Pilar | Pergunta que responde | Como implementamos |
|---|---|---|
| **Descoberta automatica** | Quais videos novos existem? | YouTube Data API v3 (sem colar ID) |
| **Incrementalidade (watermark)** | O que mudou desde a ultima vez? | Guardar o maior `publishedAt` ja visto |
| **Idempotencia** | Rodar 2x duplica? | Chave + `content_hash` + UPSERT |
| **Agendamento por negocio** | De quanto em quanto tempo? | APScheduler com intervalo por canal |

Vamos construir a intuicao de cada um com codigo executavel, usando um
**simulador em memoria** (sem precisar de API key). O projeto-script faz o
mesmo, mas com a API real e SQLite em disco.


### Setup

In [ ]:
!pip install -q pandas duckdb pandera youtube-transcript-api google-api-python-client APScheduler PyYAML
import hashlib, random
from datetime import datetime, timedelta, timezone
import pandas as pd
random.seed(42)
print("pronto")

## Pilar 1 — Descoberta automatica (e a economia de COTA)

Colar ID nao escala. A **YouTube Data API v3** lista videos de um canal
automaticamente. Mas ela tem um orcamento: **10.000 unidades/dia**. O custo
de cada chamada e o que separa o engenheiro do amador:

| Chamada | Custo | Uso |
|---|---|---|
| `search.list` | **100** unidades | NUNCA em loop de polling |
| `channels.list` | 1 unidade | 1x para achar a playlist de uploads |
| `playlistItems.list` | 1 unidade | a cada ciclo, lista uploads recentes |
| `videos.list` | 1 unidade | lote de ate 50 IDs -> metadados ricos |

**Estrategia correta:** todo canal tem uma playlist automatica de uploads
(ID comeca com `UU`). Descobrimos ela uma vez (`channels.list`) e depois
paginamos `playlistItems` (1 unidade) a cada ciclo. Um canal custa ~2
unidades/ciclo em vez de 100+. Isso permite dezenas de canais o dia inteiro.

> Decisao de engenharia: trocar `search.list` por `playlistItems.list`
> reduz o custo em **~50x**. No projeto-script isso esta em
> `ingestor/discovery.py`.


In [ ]:
# Simulacao do que a API retornaria (metadados RICOS, nao so o ID)
def descobrir_videos_simulado(channel_id, n=5):
    rng = random.Random(channel_id)
    agora = datetime.now(timezone.utc)
    videos = []
    for i in range(n):
        pub = agora - timedelta(days=rng.randint(0, 20))
        videos.append({
            "video_id": f"{channel_id[-3:]}_{i}",
            "title": f"Episodio {i}",
            "published_at": pub.isoformat(),
            "view_count": rng.randint(1000, 200000),
            "like_count": rng.randint(50, 9000),
            "duration_iso": f"PT{rng.randint(5,45)}M",
            "tags": ["economia", "mercado"],
        })
    return sorted(videos, key=lambda v: v["published_at"], reverse=True)

catalogo = descobrir_videos_simulado("UCfinancas")
pd.DataFrame(catalogo)

Repare: alem do ID, ja trouxemos **`view_count`, `like_count`, `duration`,
`tags`, `published_at`** — metadados que o YouTube libera de graca e que o
material anterior ignorava. Eles viram dimensoes analiticas riquissimas na Gold
(ex.: "densidade de termo X ponderada por views").

## Pilar 2 — Incrementalidade com **watermark**

Sem watermark, todo ciclo re-baixa tudo: desperdicio de cota e de I/O. A
**marca d'agua** e simplesmente o maior `published_at` que ja processamos
naquele canal. No proximo ciclo, ignoramos tudo que veio antes dela.

In [ ]:
watermark = {}  # channel_id -> ultimo published_at processado

def filtrar_incremental(channel_id, videos):
    corte = watermark.get(channel_id)
    novos = [v for v in videos if corte is None or v["published_at"] > corte]
    if novos:
        watermark[channel_id] = max(v["published_at"] for v in novos)
    return novos

# 1a passagem: tudo e novo
n1 = filtrar_incremental("UCfinancas", catalogo)
print(f"Ciclo 1: {len(n1)} videos novos")
# 2a passagem imediata: watermark ja cobre todos -> nada novo
n2 = filtrar_incremental("UCfinancas", catalogo)
print(f"Ciclo 2: {len(n2)} videos novos (watermark funcionando!)")

## Pilar 3 — Idempotencia (rodar 2x = rodar 1x)

Watermark filtra por data, mas e se o mesmo video aparecer de novo? Ou se a
legenda **auto-gerada** for substituida por uma **corrigida**? Precisamos de
uma fonte da verdade por video. Usamos a chave `video_id` + um `content_hash`
da transcricao:

- video ja `INGESTED` com mesmo hash -> **pula** (idempotente).
- mesmo video com hash diferente -> legenda mudou -> **reprocessa**.

No projeto-script isso e a tabela `ingestion_state` em SQLite, com `UPSERT`
(`ON CONFLICT DO ...`).

In [ ]:
estado = {}  # video_id -> {"status", "hash"}

def content_hash(texto):
    return hashlib.sha256(texto.encode()).hexdigest()[:16]

def ja_ingerido(video_id, novo_hash):
    e = estado.get(video_id)
    if not e or e["status"] != "INGESTED":
        return False
    return e["hash"] == novo_hash      # hash diferente -> reprocessa

def ingerir(video_id, transcricao):
    h = content_hash(transcricao)
    if ja_ingerido(video_id, h):
        return "PULADO (idempotencia)"
    estado[video_id] = {"status": "INGESTED", "hash": h}
    return "INGERIDO"

print(ingerir("fin_0", "fala sobre juros e inflacao"))     # INGERIDO
print(ingerir("fin_0", "fala sobre juros e inflacao"))     # PULADO
print(ingerir("fin_0", "legenda CORRIGIDA sobre selic"))   # reprocessa

## Pilar 4 — Agendamento **por negocio**

A frequencia certa depende do dominio, nao do codigo:

- Canal de **guerra/geopolitica**: pauta muda em minutos → a cada **15 min**.
- Canal de **financas**: pautas macro mais lentas → a cada **60 min**.
- Canal de **review de gadget**: sem urgencia → a cada **6 h**.

Isso vira **configuracao declarativa** (`config/canais.yaml`), e o
**APScheduler** agenda cada canal no seu intervalo. Trecho do entrypoint real
(`scheduler.py`):

In [ ]:
# Ilustracao do agendamento (nao bloqueante aqui, so demonstra a API)
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.interval import IntervalTrigger

def ciclo(nome):
    print(f"[{datetime.now():%H:%M:%S}] rodando ingestao de {nome}")

sched = BackgroundScheduler()
canais = [("Geopolitica", 15), ("Financas", 60), ("Reviews", 360)]
for nome, minutos in canais:
    sched.add_job(ciclo, IntervalTrigger(minutes=minutos),
                  args=[nome], name=nome, max_instances=1, coalesce=True)
    print(f"agendado: {nome} a cada {minutos} min")
# sched.start()  # no projeto real, BlockingScheduler mantem o processo vivo

## Juntando tudo: o projeto-script de producao

Os quatro pilares acima ja estao implementados, testados e prontos para a VPS
no pacote `youtube_ingestor/`:

```
youtube_ingestor/
├── config/canais.yaml      # canais + intervalo por negocio (declarativo)
├── ingestor/
│   ├── discovery.py        # Pilar 1: Data API v3 + estrategia de cota
│   ├── state.py            # Pilares 2 e 3: SQLite (watermark + idempotencia)
│   ├── transcript.py       # captacao da legenda (com fallback)
│   └── pipeline.py         # Bronze -> Silver(Pandera) -> Gold(DuckDB)
├── scheduler.py            # Pilar 4: APScheduler (entrypoint)
├── requirements.txt
└── .env.example            # YOUTUBE_API_KEY
```

### Como rodar
```bash
pip install -r youtube_ingestor/requirements.txt

# 1 ciclo de cada canal e sai (bom para testar / cron externo):
python -m youtube_ingestor.scheduler --once

# modo perene (fica rodando, agenda por canal):
python -m youtube_ingestor.scheduler

# ver o estado acumulado:
python -m youtube_ingestor.scheduler --status
```

Por padrao o projeto vem com `modo_demo: true` em `canais.yaml`: roda a aula
inteira **sem API key e sem gastar cota**, com dados sinteticos
deterministicos. Para producao, troque para `false`, preencha os
`channel_id` reais e a `YOUTUBE_API_KEY` no `.env`.

---

### Seu desafio (entrega)

1. Pegue **um** dos 10 dominios do desafio anterior e configure o canal real
   correspondente em `canais.yaml` (com o intervalo que faz sentido para o
   negocio — justifique a escolha).
2. Rode em `--once` 3 vezes seguidas e **prove**, consultando
   `--status` e a tabela `ingestion_state`, que a 2a e 3a execucoes
   **nao reingerem** o que ja foi processado.
3. Especialize a camada **Gold** (`pipeline.py::_gold`) com a analise do SEU
   dominio (reaproveite o SQL do notebook do desafio anterior).
4. **Bonus:** adicione um campo de metadado novo (ex.: `view_count`) e use-o
   para **ponderar** a metrica da Gold.

> Rastreabilidade: este desafio e a evolucao direta dos notebooks
> `01..10` — mesma fonte, mesma arquitetura Bronze/Silver/Gold, agora
> **governada e perene**.
